# Member 3 — bird model, observed rates and spatial checks (AviTrend PH)

Tests whether night-time light affects **light-sensitive** birds more than **light-tolerant** birds, using only real eBird checklists (true absences included, observation gaps never filled). Follows parts B and C of *Member 3* in `docs/TEAM_BUILD_PLAN.md`.

| Section | Needs | Status |
|---|---|---|
| 1–2 Load Member 1 data, traits, ≤ 2 km + hotspot thinning | Member 1 files | runs now |
| 3 Covariates + collinearity | Member 2 `grid_monthly`, `grid_landcover` | **switches on automatically when the files exist** |
| 4 Modelling table (aggregated, exact equivalent of the long model) | Member 1 | runs now |
| 5 Baseline effort-only model + sanity checks | Member 1 | runs now |
| 6 Full light model, headline numbers, effect curve, robustness | Member 2 | waits |
| 7 Observed reporting rates per cell (for the app map) | Member 1 | runs now |
| 8 Moran's I: gaps, effort, model residuals | Member 1 (+ `water_frac` for land-only) | runs now (provisional) |
| 9 `findings.md` | all | writes what is ready, marks the rest pending |

**Why an aggregated table?** In the plan's model every species of the same trait group shares the same predictors on a checklist. Counting detections per checklist × trait group (binomial: successes out of species in the group) gives **exactly the same estimates** as one row per checklist × species, but with ~70k rows instead of millions. Section 4 verifies this on a sample.

## 0. Setup and config

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
warnings.filterwarnings("ignore", category=FutureWarning)
try:
    sys.stdout.reconfigure(encoding="utf-8")      # Windows consoles: print ≤ and – without crashing
except Exception:
    pass

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
try:
    import libpysal, esda
    HAS_PYSAL = True
except ImportError:
    HAS_PYSAL = False
    print("libpysal/esda missing: in Colab run  !pip install -q libpysal esda  then restart this cell")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/NASA Hackathon")      # <- edit if needed
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROC = PROJECT_ROOT / "data" / "processed"        # Member 1 files; outputs are written here too
COV_DIR = PROC                                     # where Member 2's grid_monthly / grid_landcover will appear
FIG = PROC / "figures"

# ---- model settings (from the plan) ----
MAIN_MAX_KM = 2.0                 # main model: checklists within 2 km of their 1 km cell
THIN_MAX_PER_CELL_WEEK = 3        # hotspot thinning: max checklists per cell x year x week
SEED = 42
PERIODS = {"2018-2021": (2018, 2021), "2022-2025": (2022, 2025)}   # balanced periods for the map
MIN_CHECKLISTS_FOR_RATE = 5
WATER_MAX = 0.5
COLLINEAR_R = 0.7
STD_CHECKLIST = {"duration_min": 60, "distance_km": 1.0, "n_observers": 1, "hour_started": 7.0}
LAT_MIN, LON_MIN, CELL_DEG = 14.00, 120.50, 0.009
N_ROWS, N_COLS = 145, 123

def read(folder, name):
    pq, cs = folder / f"{name}.parquet", folder / f"{name}.csv"
    if pq.exists():
        return pd.read_parquet(pq)
    if cs.exists():
        return pd.read_csv(cs)
    return None

def save(df, name):
    path = PROC / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"saved {path.name:34s}{len(df):>10,} rows")

COVARIATES_READY = all((COV_DIR / f"{n}.parquet").exists() or (COV_DIR / f"{n}.csv").exists()
                       for n in ["grid_monthly", "grid_landcover"])
print("Member 2 covariates ready:", COVARIATES_READY)

Member 2 covariates ready: True


## 1. Load Member 1 data and map traits

In [2]:
ck = read(PROC, "checklists"); de = read(PROC, "detections"); sp = read(PROC, "species"); ef = read(PROC, "effort_grid")
for name, df in [("checklists", ck), ("detections", de), ("species", sp), ("effort_grid", ef)]:
    assert df is not None, f"missing data/processed/{name}"
de["species"] = de["species"].astype(str)

sp["sensitive"] = (sp["light_tolerance"].astype(str).str.strip().str.lower() == "sensitive").astype(int)
sp["migrant"] = (sp["migratory_status"].astype(str).str.strip().str.lower() == "migratory").astype(int)
focus = sp[sp["focus"]].copy()
print(f"checklists {len(ck):,} | detections {len(de):,} | focus species {len(focus)}")
print("\nTrait mapping (focus species):")
print(pd.crosstab(focus["light_tolerance"], focus["migratory_status"], margins=True))

checklists 23,947 | detections 3,424,421 | focus species 143

Trait mapping (focus species):
migratory_status  Migratory  Resident  All
light_tolerance                           
Sensitive                31        30   61
Tolerant                 12        70   82
All                      43       100  143


## 2. Main sample: ≤ 2 km checklists, hotspot thinning

Thinning keeps at most 3 random checklists per cell × year × week, so heavily birded sites (La Mesa, the wetlands, Candaba) don't dominate.

In [3]:
def prepare_checklists(c, max_km=MAIN_MAX_KM, thin=True, seed=SEED):
    c = c[c["distance_km"] <= max_km].copy()
    if thin:
        c = (c.sample(frac=1.0, random_state=seed)
               .groupby(["cell_id", "year", "week"], sort=False).head(THIN_MAX_PER_CELL_WEEK).copy())
    c["log_duration"] = np.log(c["duration_min"])
    c["hour2"] = c["hour_started"] ** 2
    c["year_c"] = c["year"] - 2019
    return c.dropna(subset=["hour_started"])

ck_main = prepare_checklists(ck)
print(f"all usable: {len(ck):,} | <= {MAIN_MAX_KM:g} km: {(ck['distance_km'] <= MAIN_MAX_KM).sum():,} | "
      f"after thinning: {len(ck_main):,} checklists in {ck_main['cell_id'].nunique():,} cells")
print(ck_main.groupby("year").size().rename("checklists").to_frame().T.to_string())

all usable: 23,947 | <= 2 km: 21,020 | after thinning: 15,890 checklists in 1,364 cells
year        2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025
checklists    85   244   352   353   478   775  1222  2062  1974  2167  2672  3506


## 3. Covariates (switches on when Member 2's files exist)

Joins night light, NDVI, land temperature, rain (cell × month) and land-cover shares (cell × year); drops water cells; z-scores predictors; checks collinearity and decides which habitat terms stay in.

In [4]:
COV_TERMS, cov_stats, HABITAT, sample_log = [], {}, [], {}
CANDIDATES = ["log_alan", "ndvi", "lst_day", "rain_mm", "urban_frac", "forest_frac"]
_cov_cache = {}

def zscore(df, cols):
    for col in cols:
        mu, sd = df[col].mean(), df[col].std()
        cov_stats[col] = (mu, sd)
        df["z_" + col] = (df[col] - mu) / sd
    return df

def vif_table(df, cols):
    X = df[cols].dropna().to_numpy()
    out = []
    for i, col in enumerate(cols):
        y, others = X[:, i], np.delete(X, i, axis=1)
        A = np.c_[np.ones(len(others)), others]
        resid = y - A @ np.linalg.lstsq(A, y, rcond=None)[0]
        r2 = 1 - resid.var() / y.var()
        out.append({"variable": col, "VIF": 1 / (1 - r2) if r2 < 1 else np.inf})
    return pd.DataFrame(out)

def cov_tables():
    if not _cov_cache:
        gm = read(COV_DIR, "grid_monthly")
        _cov_cache["gm"] = gm[["cell_id", "year", "month", "alan", "ndvi", "lst_day", "rain_mm", "water_frac"]]
        _cov_cache["lc"] = read(COV_DIR, "grid_landcover")[["cell_id", "year", "urban_frac", "forest_frac"]]
    return _cov_cache["gm"], _cov_cache["lc"]

def merge_covariates(c, keep_water=False):
    """Join cell x month covariates and cell x year land cover; drop water-heavy cells unless keep_water."""
    gm, lc = cov_tables()
    x = c.merge(gm, on=["cell_id", "year", "month"], how="left").merge(lc, on=["cell_id", "year"], how="left")
    n0 = len(x)
    if not keep_water:
        x = x[~(x["water_frac"] > WATER_MAX)].copy()
    x["log_alan"] = np.log1p(x["alan"])
    print(f"  {n0:,} checklists; water-heavy cells (water_frac > {WATER_MAX}) "
          f"{'kept' if keep_water else f'dropped: {n0 - len(x):,}'}")
    return x

def add_covariates(c, keep_water=False, terms=None):
    """merge_covariates, then drop rows missing a covariate that is actually in the model."""
    terms = COV_TERMS if terms is None else terms
    x = merge_covariates(c, keep_water)
    n1 = len(x)
    x = x.dropna(subset=terms)
    print(f"  dropped {n1 - len(x):,} missing a model covariate -> {len(x):,} checklists")
    return x

if COVARIATES_READY:
    n_before = len(ck_main)
    x = merge_covariates(ck_main)
    sample_log["thinned"], sample_log["after_water"] = n_before, len(x)
    print("\nMissing share among land checklists:"); print(x[CANDIDATES].isna().mean().round(3).to_string())
    corr = x[CANDIDATES].corr()                      # pairwise-complete correlations
    print("\nCorrelation matrix:"); print(corr.round(2).to_string())
    print("\nVIF, all candidates (rows with every candidate present):")
    print(vif_table(x, CANDIDATES).round(2).to_string(index=False))

    # decide the terms FIRST, then drop rows missing only those terms (NDVI is ~45% missing in the sample)
    HABITAT = ["forest_frac"]
    r_ndvi = abs(corr.loc["log_alan", "ndvi"])
    if r_ndvi <= COLLINEAR_R:
        HABITAT.append("ndvi")
    else:
        print(f"\n|r(ALAN, NDVI)| = {r_ndvi:.2f} > {COLLINEAR_R}: NDVI left out (keep ALAN)")
    print(f"|r(ALAN, urban)| = {abs(corr.loc['log_alan', 'urban_frac']):.2f}: urban share left out by design "
          "(it duplicates ALAN); forest share is the habitat term")
    print(f"|r(ALAN, LST)| = {abs(corr.loc['log_alan', 'lst_day']):.2f}: below {COLLINEAR_R}, LST kept "
          "(a 'without LST' refit is in the robustness checks)")
    COV_TERMS = ["log_alan"] + HABITAT + ["lst_day", "rain_mm"]
    ck_main = x.dropna(subset=COV_TERMS).copy()
    sample_log["model"] = len(ck_main)
    print(f"\nmodel sample: {n_before:,} thinned -> {sample_log['after_water']:,} after water rule -> "
          f"{len(ck_main):,} with all model covariates ({', '.join(COV_TERMS)})")
    ck_main = zscore(ck_main, COV_TERMS)
    print("\nVIF, model covariates:"); print(vif_table(ck_main, COV_TERMS).round(2).to_string(index=False))
else:
    print("Waiting for Member 2 (grid_monthly + grid_landcover). Sections 3 and 6 are skipped for now.")

  15,890 checklists; water-heavy cells (water_frac > 0.5) dropped: 1,831

Missing share among land checklists:
log_alan       0.001
ndvi           0.446
lst_day        0.078
rain_mm        0.000
urban_frac     0.000
forest_frac    0.000

Correlation matrix:
             log_alan  ndvi  lst_day  rain_mm  urban_frac  forest_frac
log_alan         1.00 -0.77     0.69     0.00        0.83        -0.60
ndvi            -0.77  1.00    -0.67     0.20       -0.73         0.55
lst_day          0.69 -0.67     1.00     0.01        0.70        -0.54
rain_mm          0.00  0.20     0.01     1.00        0.03        -0.02
urban_frac       0.83 -0.73     0.70     0.03        1.00        -0.45
forest_frac     -0.60  0.55    -0.54    -0.02       -0.45         1.00

VIF, all candidates (rows with every candidate present):
   variable  VIF
   log_alan 3.86
       ndvi 3.24
    lst_day 2.28
    rain_mm 1.07
 urban_frac 3.27
forest_frac 1.73

|r(ALAN, NDVI)| = 0.77 > 0.7: NDVI left out (keep ALAN)
|r(ALAN, ur

## 4. Modelling table: detections per checklist × trait group

`successes` = focus species of that group detected on the checklist, `trials` = focus species in that group. Verified below to give the same coefficients as the long one-row-per-species table.

In [5]:
grp_cols = focus[["species", "sensitive", "migrant"]]

def build_group_table(c):
    d = de[de["checklist_id"].isin(c["checklist_id"])].merge(grp_cols, on="species")
    g = (d.groupby(["checklist_id", "sensitive", "migrant"], as_index=False)
           .agg(successes=("detected", "sum"), trials=("detected", "size")))
    g["failures"] = g["trials"] - g["successes"]
    return g.merge(c, on="checklist_id")

g_main = build_group_table(ck_main)
g_main["cluster"] = pd.factorize(g_main["cell_id"])[0]
print(f"modelling rows: {len(g_main):,} (checklists {g_main['checklist_id'].nunique():,} x up to 4 trait groups)")
print(g_main.groupby(["sensitive", "migrant"])[["successes", "trials"]].sum()
      .assign(rate=lambda t: t["successes"] / t["trials"]).round(3).to_string())

# --- equivalence check on a sample: aggregated binomial == long logistic ---
EFFORT = "log_duration + distance_km + n_observers + hour_started + hour2"
f_check = f"~ {EFFORT} + sensitive + migrant + year_c"
samp_ids = ck_main["checklist_id"].sample(min(1500, len(ck_main)), random_state=SEED)
long = (de[de["checklist_id"].isin(samp_ids)].merge(grp_cols, on="species").merge(ck_main, on="checklist_id"))
m_long = smf.glm("detected " + f_check, long, family=sm.families.Binomial()).fit()
m_agg = smf.glm("successes + failures " + f_check, g_main[g_main["checklist_id"].isin(samp_ids)],
                family=sm.families.Binomial()).fit()
diff = (m_long.params - m_agg.params).abs().max()
print(f"\nequivalence check: max coefficient difference long vs aggregated = {diff:.2e}  "
      f"[{'PASS' if diff < 1e-6 else 'FAIL'}]")

modelling rows: 51,812 (checklists 12,953 x up to 4 trait groups)
                   successes  trials   rate
sensitive migrant                          
0         0           109407  906710  0.121
          1             4627  155436  0.030
1         0            10184  388590  0.026
          1            11853  401543  0.030

equivalence check: max coefficient difference long vs aggregated = 2.84e-14  [PASS]


## 5. Baseline model: effort + season + year (runs now)

No covariates yet. Validates the data and gives effort-corrected trends per trait group. Logistic regression (binomial GLM) with **cluster-robust standard errors by cell** (repeat visits to the same place are not independent).

In [6]:
def fit(formula, g):
    return smf.glm(formula, g, family=sm.families.Binomial()).fit(
        cov_type="cluster", cov_kwds={"groups": pd.factorize(g["cell_id"])[0]})

def coef_table(m):
    ci = m.conf_int()
    return pd.DataFrame({"term": m.params.index, "coef": m.params.values, "odds_ratio": np.exp(m.params.values),
                         "or_ci_low": np.exp(ci[0].values), "or_ci_high": np.exp(ci[1].values),
                         "p_value": m.pvalues.values})

def lincomb(m, expr):
    t = m.t_test(expr)
    est, (lo, hi) = float(np.squeeze(t.effect)), np.squeeze(t.conf_int())
    return {"odds_ratio": np.exp(est), "ci_low": np.exp(lo), "ci_high": np.exp(hi), "p_value": float(np.squeeze(t.pvalue))}

f_base = f"successes + failures ~ {EFFORT} + C(month) + year_c * sensitive + migrant"
m_base = fit(f_base, g_main)
base_tab = coef_table(m_base)
base_tab.to_csv(PROC / "model_coefficients_baseline.csv", index=False)
print(base_tab[~base_tab["term"].str.startswith("C(month)")].round(4).to_string(index=False))

checks = []
def check(name, ok, detail=""):
    checks.append((name, bool(ok))); print(f"[{'PASS' if ok else 'FAIL'}] {name} {detail}")

b = m_base.params
check("log_duration coefficient is positive (longer checklists detect more)", b["log_duration"] > 0,
      f"(coef {b['log_duration']:.3f})")
check("model converged, no NaN coefficients", m_base.converged and not b.isna().any())

seas = (g_main.assign(season=np.where(g_main["month"].isin([10, 11, 12, 1, 2, 3]), "Oct-Mar", "Apr-Sep"))
              .groupby(["migrant", "season"])[["successes", "trials"]].sum())
seas["rate"] = seas["successes"] / seas["trials"]
print("\nRaw reporting rate by season:"); print(seas["rate"].unstack().round(3).to_string())
check("migrants reported more in Oct-Mar", seas.loc[(1, "Oct-Mar"), "rate"] > seas.loc[(1, "Apr-Sep"), "rate"])

trend_tol = lincomb(m_base, "year_c = 0")
trend_sen = lincomb(m_base, "year_c + year_c:sensitive = 0")
print(f"\nEffort-corrected trend (odds of detection per year): tolerant {trend_tol['odds_ratio']:.3f} "
      f"[{trend_tol['ci_low']:.3f}, {trend_tol['ci_high']:.3f}], sensitive {trend_sen['odds_ratio']:.3f} "
      f"[{trend_sen['ci_low']:.3f}, {trend_sen['ci_high']:.3f}]  (2014-2016 data are sparse: interpret with care)")

            term    coef  odds_ratio  or_ci_low  or_ci_high  p_value
       Intercept -3.2082      0.0404     0.0244      0.0670   0.0000
    log_duration  0.4157      1.5154     1.4363      1.5989   0.0000
     distance_km  0.1651      1.1795     1.0948      1.2707   0.0000
     n_observers  0.0173      1.0175     0.9915      1.0442   0.1892
    hour_started -0.0840      0.9194     0.8517      0.9925   0.0314
           hour2  0.0027      1.0027     0.9996      1.0059   0.0911
          year_c  0.0169      1.0170     1.0018      1.0324   0.0283
       sensitive -1.1772      0.3082     0.2588      0.3669   0.0000
year_c:sensitive -0.0152      0.9849     0.9618      1.0086   0.2110
         migrant -0.7531      0.4709     0.4160      0.5330   0.0000
[PASS] log_duration coefficient is positive (longer checklists detect more) (coef 0.416)
[PASS] model converged, no NaN coefficients 

Raw reporting rate by season:
season   Apr-Sep  Oct-Mar
migrant                  
0          0.096    0.08

## 6. Full light model (waits for Member 2)

The plan's model: `z_alan × sensitive` is the key term. Headline numbers, effect curve, robustness refits.

In [7]:
full_results, robustness = None, None
if COVARIATES_READY:
    cov_formula = " + ".join("z_" + v for v in COV_TERMS if v != "log_alan")
    f_full = (f"successes + failures ~ z_log_alan * sensitive + {cov_formula} + {EFFORT}"
              f" + C(month) + year_c * sensitive + migrant")
    m_full = fit(f_full, g_main)

    sd_alan = cov_stats["log_alan"][1]
    tol = lincomb(m_full, "z_log_alan = 0")
    sen = lincomb(m_full, "z_log_alan + z_log_alan:sensitive = 0")
    p_int = m_full.pvalues["z_log_alan:sensitive"]

    full_tab = coef_table(m_full)
    full_tab["label"] = None                      # the app labels known terms itself
    derived = pd.DataFrame([{"term": "z_log_alan (sensitive)", "coef": np.log(sen["odds_ratio"]),
                             "odds_ratio": sen["odds_ratio"], "or_ci_low": sen["ci_low"], "or_ci_high": sen["ci_high"],
                             "p_value": sen["p_value"], "label": "Per 1 SD more night light, light-sensitive species"}])
    full_tab = pd.concat([full_tab, derived], ignore_index=True)
    full_tab.to_csv(PROC / "model_coefficients.csv", index=False)
    print(full_tab[~full_tab["term"].str.startswith("C(month)")].drop(columns="label").round(4).to_string(index=False))

    full_results = {"or_tolerant": tol, "or_sensitive": sen, "p_interaction": p_int,
                    "n_checklists": g_main["checklist_id"].nunique(), "n_cells": g_main["cell_id"].nunique()}
    print(f"\nPer 1 SD more night light (x{np.exp(sd_alan):.2f} in radiance+1):")
    print(f"  tolerant species:  odds of detection x{tol['odds_ratio']:.3f} [{tol['ci_low']:.3f}, {tol['ci_high']:.3f}]")
    print(f"  sensitive species: odds of detection x{sen['odds_ratio']:.3f} [{sen['ci_low']:.3f}, {sen['ci_high']:.3f}]")
    print(f"  difference between groups (interaction): p = {p_int:.4f}")
    check("full model converged, no NaN coefficients", m_full.converged and not m_full.params.isna().any(),
          f"({full_results['n_checklists']:,} checklists, {full_results['n_cells']:,} cells, {len(g_main):,} rows)")

    # ---- effect curve: detection probability vs night light, standard checklist, averaged over months ----
    q = ck_main["log_alan"].quantile(np.linspace(0.05, 0.95, 40)).to_numpy()
    base_row = {**STD_CHECKLIST, "hour2": STD_CHECKLIST["hour_started"] ** 2,
                "log_duration": np.log(STD_CHECKLIST["duration_min"]), "year_c": 2025 - 2019, "migrant": 0}
    for v in COV_TERMS:
        if v != "log_alan":
            base_row["z_" + v] = 0.0                                   # other covariates at their mean
    rows = []
    for s_ in (0, 1):
        for la in q:
            for mth in range(1, 13):
                rows.append({**base_row, "sensitive": s_, "month": mth, "log_alan": la,
                             "z_log_alan": (la - cov_stats["log_alan"][0]) / sd_alan})
    grid = pd.DataFrame(rows)
    pr = m_full.get_prediction(grid).summary_frame()
    grid[["prob", "ci_low", "ci_high"]] = pr[["mean", "mean_ci_lower", "mean_ci_upper"]].to_numpy()
    effect = grid.groupby(["sensitive", "log_alan"], as_index=False)[["prob", "ci_low", "ci_high"]].mean()
    effect["alan"] = np.expm1(effect["log_alan"])
    effect["group"] = np.where(effect["sensitive"] == 1, "Light-sensitive", "Light-tolerant")
    save(effect[["group", "sensitive", "alan", "prob", "ci_low", "ci_high"]], "effect_curve")
    if HAS_MPL:
        fig, ax = plt.subplots(figsize=(7, 4))
        for name, e in effect.groupby("group"):
            ax.plot(e["alan"], e["prob"], label=name); ax.fill_between(e["alan"], e["ci_low"], e["ci_high"], alpha=0.2)
        ax.set_xlabel("Night-time light (nW/cm²/sr)"); ax.set_ylabel("Detection probability (standard checklist)")
        ax.set_title("Predicted detection vs night light (resident species, 2025, 60-min checklist)"); ax.legend()
        FIG.mkdir(parents=True, exist_ok=True); fig.savefig(FIG / "effect_curve.png", dpi=150, bbox_inches="tight"); plt.show()

    # ---- raw cross-check: observed reporting rate by night-light quartile, per group ----
    gq = g_main.assign(alan_q=pd.qcut(g_main["log_alan"], 4, labels=["Q1 darkest", "Q2", "Q3", "Q4 brightest"]))
    xq = gq.groupby(["sensitive", "alan_q"], observed=True)[["successes", "trials"]].sum()
    xq["rate"] = xq["successes"] / xq["trials"]
    quart = xq["rate"].unstack()
    quart.index = ["Light-tolerant" if i == 0 else "Light-sensitive" for i in quart.index]
    quart.to_csv(PROC / "alan_quartile_check.csv")
    print("\nObserved reporting rate by night-light quartile (main sample, no model):"); print(quart.round(4).to_string())
    agree = []
    for s_, name in ((0, "Light-tolerant"), (1, "Light-sensitive")):
        e = effect[effect["sensitive"] == s_].sort_values("alan")
        agree.append(np.sign(quart.loc[name].iloc[-1] - quart.loc[name].iloc[0]) == np.sign(e["prob"].iloc[-1] - e["prob"].iloc[0]))
    check("raw ALAN-quartile rates point the same way as the effect curve (both groups)", all(agree),
          "(raw rates are not effort-corrected; disagreement means confounding by effort/season/habitat)")

    # ---- does the sensitive-vs-tolerant difference hold for both residents and migrants? ----
    f_mig = f_full.replace("z_log_alan * sensitive", "z_log_alan * sensitive * migrant")
    m_mig = fit(f_mig, g_main)
    combos = {("Resident", "Light-tolerant"): "z_log_alan",
              ("Resident", "Light-sensitive"): "z_log_alan + z_log_alan:sensitive",
              ("Migrant", "Light-tolerant"): "z_log_alan + z_log_alan:migrant",
              ("Migrant", "Light-sensitive"): "z_log_alan + z_log_alan:sensitive + z_log_alan:migrant"
                                              " + z_log_alan:sensitive:migrant"}
    by_status = pd.DataFrame([{"status": k[0], "group": k[1], **lincomb(m_mig, e + " = 0")} for k, e in combos.items()])
    diff_p = {"Resident": m_mig.pvalues["z_log_alan:sensitive"],
              "Migrant": lincomb(m_mig, "z_log_alan:sensitive + z_log_alan:sensitive:migrant = 0")["p_value"]}
    by_status["p_sensitive_vs_tolerant"] = by_status["status"].map(diff_p)
    by_status.to_csv(PROC / "alan_by_migratory_status.csv", index=False)
    full_results["by_status"] = by_status
    print("\nPer 1 SD more night light, by migratory status (model with z_log_alan x sensitive x migrant):")
    print(by_status.round(4).to_string(index=False))

    # ---- robustness refits ----
    def interaction_of(g, formula):
        f = formula if g["migrant"].nunique() > 1 else formula.replace(" + migrant", "")   # constant term -> drop it
        m = fit(f, g)
        return m.params["z_log_alan:sensitive"], m.pvalues["z_log_alan:sensitive"], g["checklist_id"].nunique()
    f_no_lst = f_full.replace(" + z_lst_day", "")
    print("\nBuilding robustness samples:")
    variants = {
        "main (<= 2 km, thinned)": (g_main, f_full),
        "no thinning": (build_group_table(add_covariates(prepare_checklists(ck, thin=False))), f_full),
        "residents only": (g_main[g_main["migrant"] == 0].copy(), f_full),
        "<= 5 km": (build_group_table(add_covariates(prepare_checklists(ck, max_km=5.0))), f_full),
        "incl. water-heavy cells": (build_group_table(add_covariates(prepare_checklists(ck), keep_water=True)), f_full),
        "without LST": (g_main, f_no_lst),
    }
    rob = []
    for name, (g, formula) in variants.items():
        for v in COV_TERMS:                       # re-apply the main sample's z-scores
            if "z_" + v not in g:
                g["z_" + v] = (g[v] - cov_stats[v][0]) / cov_stats[v][1]
        coef, p, n = interaction_of(g, formula)
        rob.append({"variant": name, "interaction_coef": coef, "interaction_or": np.exp(coef), "p_value": p,
                    "n_checklists": n})
    robustness = pd.DataFrame(rob)
    robustness.to_csv(PROC / "robustness.csv", index=False)
    print("\nRobustness of the key term (z_log_alan:sensitive):"); print(robustness.round(4).to_string(index=False))
    same_sign = (np.sign(robustness["interaction_coef"]) == np.sign(robustness["interaction_coef"].iloc[0])).all()
    check("key term keeps its sign in every robustness variant", same_sign)
else:
    print("Skipped: waiting for Member 2's covariates.")

                  term    coef  odds_ratio  or_ci_low  or_ci_high  p_value
             Intercept -3.1505      0.0428     0.0260      0.0705   0.0000
            z_log_alan -0.1247      0.8827     0.8226      0.9473   0.0005
             sensitive -1.4096      0.2443     0.2035      0.2932   0.0000
  z_log_alan:sensitive -0.5094      0.6009     0.5437      0.6641   0.0000
         z_forest_frac -0.1900      0.8270     0.7650      0.8940   0.0000
             z_lst_day  0.0765      1.0795     0.9859      1.1820   0.0985
             z_rain_mm -0.0150      0.9851     0.9512      1.0202   0.4004
          log_duration  0.4001      1.4920     1.4197      1.5681   0.0000
           distance_km  0.1565      1.1694     1.1008      1.2423   0.0000
           n_observers  0.0188      1.0190     0.9975      1.0408   0.0831
          hour_started -0.0877      0.9160     0.8519      0.9850   0.0180
                 hour2  0.0029      1.0029     0.9999      1.0059   0.0573
                year_c  0

## 7. Observed reporting rates per cell (for the app map)

Observed data only, no model. Rate = detections ÷ (checklists × focus species in the group). Cells with fewer than 5 checklists in a period are left out (they stay outlined as low-data on the map).

In [8]:
ck_rate = ck[ck["distance_km"] <= 5.0]                 # all usable checklists, unthinned
d = de[de["checklist_id"].isin(ck_rate["checklist_id"])].merge(grp_cols, on="species")
per = d.groupby(["checklist_id", "sensitive"], as_index=False).agg(successes=("detected", "sum"), trials=("detected", "size"))
per = per.merge(ck_rate[["checklist_id", "cell_id", "cell_row", "cell_col", "year"]], on="checklist_id")
rows = []
for label, (y0, y1) in PERIODS.items():
    sub = per[per["year"].between(y0, y1)]
    agg = sub.groupby(["cell_id", "cell_row", "cell_col", "sensitive"], as_index=False).agg(
        n_checklists=("checklist_id", "nunique"), successes=("successes", "sum"), trials=("trials", "sum"))
    agg["period"] = label
    rows.append(agg)
cell_rates = pd.concat(rows, ignore_index=True)
cell_rates = cell_rates[cell_rates["n_checklists"] >= MIN_CHECKLISTS_FOR_RATE].copy()
cell_rates["reporting_rate"] = cell_rates["successes"] / cell_rates["trials"]
cell_rates["group"] = np.where(cell_rates["sensitive"] == 1, "Light-sensitive", "Light-tolerant")
cell_rates["lat"] = LAT_MIN + (cell_rates["cell_row"] + 0.5) * CELL_DEG
cell_rates["lon"] = LON_MIN + (cell_rates["cell_col"] + 0.5) * CELL_DEG
cell_rates = cell_rates[["cell_id", "lat", "lon", "period", "group", "sensitive", "n_checklists", "reporting_rate"]]
save(cell_rates, "cell_rates")
print(cell_rates.groupby(["period", "group"]).agg(cells=("cell_id", "nunique"),
      median_rate=("reporting_rate", "median")).round(3).to_string())

saved cell_rates.parquet                       976 rows
                           cells  median_rate
period    group                              
2018-2021 Light-sensitive    180        0.022
          Light-tolerant     180        0.116
2022-2025 Light-sensitive    308        0.019
          Light-tolerant     308        0.095


## 8. Moran's I: gaps, effort and model residuals

Gaps and effort justify keeping gaps blank and thinning hotspots. Residuals check whether leftover spatial clustering makes the p-values too optimistic. Land-only once Member 2's `water_frac` exists.

In [9]:
spatial = []
if HAS_PYSAL:
    cell = ef.groupby(["cell_row", "cell_col"])["n_checklists"].sum()
    full = pd.DataFrame({"n": cell}).reset_index()
    land_note = "all cells (provisional: includes water)"
    gm = cov_tables()[0] if COVARIATES_READY else None
    if gm is not None and "water_frac" in gm:
        water = gm.drop_duplicates("cell_id")[["cell_id", "water_frac"]]
        rc = water["cell_id"].str.split("_", expand=True).astype(int)
        water = water.assign(cell_row=rc[0], cell_col=rc[1])
        full = full.merge(water[["cell_row", "cell_col", "water_frac"]], on=["cell_row", "cell_col"], how="left")
        full = full[~(full["water_frac"] > WATER_MAX)]
        land_note = "land cells only"
    coords = full[["cell_col", "cell_row"]].to_numpy()
    w = libpysal.weights.KNN.from_array(coords, k=8); w.transform = "r"
    m_gap = esda.Moran((full["n"] == 0).astype(float).to_numpy(), w, permutations=999)
    spatial.append({"test": "gaps", "cells": len(full), "scope": land_note, "moran_i": m_gap.I, "p_sim": m_gap.p_sim})

    obs = full[full["n"] > 0]
    wk = libpysal.weights.KNN.from_array(obs[["cell_col", "cell_row"]].to_numpy(), k=8); wk.transform = "r"
    m_eff = esda.Moran(np.log1p(obs["n"].to_numpy()), wk, permutations=999)
    spatial.append({"test": "effort", "cells": len(obs), "scope": "observed cells", "moran_i": m_eff.I, "p_sim": m_eff.p_sim})

    def residual_moran(model, g, label):
        r = g.assign(resid=model.resid_pearson).groupby(["cell_id", "cell_row", "cell_col"]).agg(
            resid=("resid", "mean"), n=("checklist_id", "nunique")).reset_index()
        r = r[r["n"] >= 5]
        wr = libpysal.weights.KNN.from_array(r[["cell_col", "cell_row"]].to_numpy(), k=8); wr.transform = "r"
        mr = esda.Moran(r["resid"].to_numpy(), wr, permutations=999)
        spatial.append({"test": f"residuals ({label})", "cells": len(r), "scope": "cells with >= 5 checklists",
                        "moran_i": mr.I, "p_sim": mr.p_sim})
        return mr

    residual_moran(m_base, g_main, "baseline model")
    if full_results is not None:
        mr = residual_moran(m_full, g_main, "full model")
        if mr.p_sim < 0.05:
            g_blk = g_main.assign(block=(g_main["cell_row"] // 5).astype(str) + "_" + (g_main["cell_col"] // 5).astype(str))
            m_blk = smf.glm(f_full, g_blk, family=sm.families.Binomial()).fit(
                cov_type="cluster", cov_kwds={"groups": pd.factorize(g_blk["block"])[0]})
            full_results["p_interaction_block_clusters"] = m_blk.pvalues["z_log_alan:sensitive"]
            print(f"residuals clustered -> refit with 5x5-cell block clusters: interaction p = "
                  f"{full_results['p_interaction_block_clusters']:.4f}")
    spatial = pd.DataFrame(spatial)
    # the app shows the first 'residuals' row: put the full model's ahead of the baseline's
    spatial = spatial.iloc[spatial["test"].eq("residuals (baseline model)").argsort(kind="stable")].reset_index(drop=True)
    spatial.to_csv(PROC / "spatial_checks.csv", index=False)
    print(spatial.round(4).to_string(index=False))
    print("note: with 999 permutations the smallest possible p is 0.001 -> report as 'p <= 0.001'")
else:
    print("Install libpysal and esda to run this section.")

residuals clustered -> refit with 5x5-cell block clusters: interaction p = 0.0000
                      test  cells                      scope  moran_i  p_sim
                      gaps  13279            land cells only   0.2851  0.001
                    effort   1268             observed cells   0.2307  0.001
    residuals (full model)    321 cells with >= 5 checklists   0.2151  0.001
residuals (baseline model)    321 cells with >= 5 checklists   0.3074  0.001
note: with 999 permutations the smallest possible p is 0.001 -> report as 'p <= 0.001'


## 9. Write `findings.md`

In [10]:
def fmt_p(p):
    return "p ≤ 0.001" if p <= 0.001 else f"p = {p:.3f}"

sp_df = pd.DataFrame(spatial) if len(spatial) else pd.DataFrame(columns=["test"])
def moran_row(test):
    r = sp_df[sp_df["test"] == test]
    return r.iloc[0] if len(r) else None

lines = ["# Findings (Member 3)", "",
         f"Data: {g_main['checklist_id'].nunique():,} eBird checklists (≤ {MAIN_MAX_KM:g} km, thinned) in "
         f"{g_main['cell_id'].nunique():,} cells of Greater Manila, 2014–2025; {len(focus)} focus species "
         f"({int(focus['sensitive'].sum())} light-sensitive, {int((1 - focus['sensitive']).sum())} light-tolerant).", ""]
if full_results:
    tol, sen = full_results["or_tolerant"], full_results["or_sensitive"]
    p_blk = full_results.get("p_interaction_block_clusters")
    head = (f"Per 1 SD more night-time light (about {np.exp(cov_stats['log_alan'][1]):.0f}× brighter), the odds of "
            f"detecting a light-sensitive species change by ×{sen['odds_ratio']:.2f} (95% CI {sen['ci_low']:.2f}–"
            f"{sen['ci_high']:.2f}), versus ×{tol['odds_ratio']:.2f} ({tol['ci_low']:.2f}–{tol['ci_high']:.2f}) for "
            f"light-tolerant species. Difference between groups: {fmt_p(full_results['p_interaction'])} "
            f"(CIs and p-values clustered by cell).")
    bs = full_results.get("by_status")
    if bs is not None:
        orx = lambda s_, g_: bs[(bs["status"] == s_) & (bs["group"] == g_)].iloc[0]["odds_ratio"]
        p_r = bs.loc[bs["status"] == "Resident", "p_sensitive_vs_tolerant"].iloc[0]
        p_m = bs.loc[bs["status"] == "Migrant", "p_sensitive_vs_tolerant"].iloc[0]
        head += (f" The difference comes from resident species: ×{orx('Resident', 'Light-sensitive'):.2f} for resident "
                 f"light-sensitive vs ×{orx('Resident', 'Light-tolerant'):.2f} for resident light-tolerant species "
                 f"({fmt_p(p_r)}), while migratory light-sensitive and light-tolerant species "
                 + ("respond alike" if p_m >= 0.05 else "differ much less")
                 + f" (×{orx('Migrant', 'Light-sensitive'):.2f} vs ×{orx('Migrant', 'Light-tolerant'):.2f}, {fmt_p(p_m)}).")
    if p_blk is not None:
        head += (f" Model residuals are spatially clustered, so with larger 5×5-cell block clusters the "
                 f"difference has {fmt_p(p_blk)}; this is the cautious figure to quote.")
    lines += ["## Headline", "", head, ""]
    lines += ["## Sample", "",
              f"- {sample_log['thinned']:,} thinned checklists → {sample_log['after_water']:,} after dropping cells "
              f"more than {WATER_MAX:.0%} water → {sample_log['model']:,} with every model covariate.",
              f"- Model covariates: {', '.join(COV_TERMS)}. NDVI left out if |r| with night light > {COLLINEAR_R}; "
              "urban share left out (duplicates night light); forest share is the habitat term.", ""]
    if robustness is not None:
        lines += ["## Robustness", "", "```", robustness.round(4).assign(p_value=robustness["p_value"].map(lambda v: f"{v:.1e}")).to_string(index=False), "```", ""]
else:
    lines += ["## Headline", "", "_Pending: needs Member 2's covariates (grid_monthly, grid_landcover)._", ""]
lines += ["## Data checks", ""] + [f"- {'PASS' if ok else 'FAIL'}: {n}" for n, ok in checks] + [""]
lines += [f"- Effort-corrected detection trend per year (baseline model): tolerant ×{trend_tol['odds_ratio']:.3f} "
          f"({trend_tol['ci_low']:.3f}–{trend_tol['ci_high']:.3f}), sensitive ×{trend_sen['odds_ratio']:.3f} "
          f"({trend_sen['ci_low']:.3f}–{trend_sen['ci_high']:.3f}). 2014–2016 data are sparse.", ""]
if len(spatial):
    lines += ["## Spatial checks (Moran's I, 999 permutations)", ""]
    for _, r in sp_df.iterrows():
        lines.append(f"- {r['test']}: I = {r['moran_i']:.3f}, {fmt_p(r['p_sim'])} ({r['cells']:,} cells, {r['scope']})")
    gap, eff = moran_row("gaps"), moran_row("effort")
    if gap is not None and eff is not None:
        lines += ["", "For the app and slides:", "",
                  f"- Unobserved cells are spatially clustered (Moran's I = {gap['moran_i']:.2f}, {fmt_p(gap['p_sim'])}, "
                  f"{gap['scope']}), so missing data is not random; we keep them as observation gaps rather than filling them.",
                  f"- Birdwatching effort is also clustered (Moran's I = {eff['moran_i']:.2f}, {fmt_p(eff['p_sim'])}), "
                  "which is why we thin hotspot checklists and cluster standard errors by cell."]
    lines.append("")
lines += ["## Caveats", "",
          "- Association, not causation.",
          "- eBird detection is not abundance; effort (duration, distance, observers, time of day) is modelled.",
          "- Results cover Greater Manila only. 88% of checklists are from 2020–2025.",
          f"- Cells more than {WATER_MAX:.0%} water are left out of the main model (their satellite values describe the bay); "
          "this removes the main wetland and mudflat hotspots. A refit that keeps water-heavy cells only partly restores "
          "them (many lack land-temperature values) and gives the same result (robustness table).",
          "- Daytime land temperature (MODIS Terra) falls after 2020 almost everywhere; this most likely reflects "
          "Terra's drifting, earlier overpass time rather than real cooling, so LST trends after 2020 should be read "
          "with care. A refit without LST is in the robustness table.",
          "- Nocturnal species, potentially the most affected by light, are under-sampled by daytime birding and "
          "mostly fall below the focus threshold, so effects are likely underestimated.",
          "- Observation gaps are never filled; statistics use real checklists only."]
(PROC / "findings.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))
print(f"\nChecks: {sum(ok for _, ok in checks)}/{len(checks)} passed")

# Findings (Member 3)

Data: 12,953 eBird checklists (≤ 2 km, thinned) in 1,136 cells of Greater Manila, 2014–2025; 143 focus species (61 light-sensitive, 82 light-tolerant).

## Headline

Per 1 SD more night-time light (about 3× brighter), the odds of detecting a light-sensitive species change by ×0.53 (95% CI 0.47–0.59), versus ×0.88 (0.82–0.95) for light-tolerant species. Difference between groups: p ≤ 0.001 (CIs and p-values clustered by cell). The difference comes from resident species: ×0.35 for resident light-sensitive vs ×0.89 for resident light-tolerant species (p ≤ 0.001), while migratory light-sensitive and light-tolerant species respond alike (×0.74 vs ×0.77, p = 0.700). Model residuals are spatially clustered, so with larger 5×5-cell block clusters the difference has p ≤ 0.001; this is the cautious figure to quote.

## Sample

- 15,890 thinned checklists → 14,059 after dropping cells more than 50% water → 12,953 with every model covariate.
- Model covariates: log_alan, for